# Credit Card Fraud Detection - Decision Tree

Non-linear tree classification with depth pruning and feature importance analysis for fraud detection.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Read dataset
try:
    df = pd.read_csv("../creditcard.csv")
    print("✅ Loaded ../creditcard.csv")
except FileNotFoundError:
    df = pd.read_csv("creditcard.csv")
    print("✅ Loaded creditcard.csv")

In [ ]:
df.columns

In [ ]:
df.head(3)

In [ ]:
# Train-test split
scaler = StandardScaler()
df_proc = df.copy()
df_proc['Amount_Scaled'] = scaler.fit_transform(df_proc[['Amount']])
df_proc.drop(columns=['Time', 'Amount'], inplace=True)

X = df_proc.drop(columns=['Class'])
y = df_proc['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training instances: {X_train.shape[0]:,}, Test instances: {X_test.shape[0]:,}")

## 1. Unconstrained Decision Tree (Prone to Overfitting)

In [ ]:
# Unconstrained baseline tree
dt_unconstrained = DecisionTreeClassifier(random_state=42)
dt_unconstrained.fit(X_train, y_train)

train_f1 = f1_score(y_train, dt_unconstrained.predict(X_train))
test_f1 = f1_score(y_test, dt_unconstrained.predict(X_test))

print("Unconstrained Tree:")
print(f"Train F1-Score: {train_f1:.4f} (100% memorization)")
print(f"Test F1-Score:  {test_f1:.4f}")
print(f"Tree Depth:     {dt_unconstrained.get_depth()}")
print(f"Leaf Nodes:     {dt_unconstrained.get_n_leaves()}")

## 2. Pruned & Balanced Decision Tree (`max_depth=5`, `class_weight='balanced'`)

In [ ]:
# Constrained tree with class weights
dt_balanced = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=50,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=42
)
dt_balanced.fit(X_train, y_train)

y_pred_dt = dt_balanced.predict(X_test)
y_prob_dt = dt_balanced.predict_proba(X_test)[:, 1]

print("=== Pruned & Balanced Decision Tree ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_dt):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_dt):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_dt):.4f}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_dt)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', cbar=False)
plt.title(f'Decision Tree Confusion Matrix (Recall: {recall_score(y_test, y_pred_dt):.2%})')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks([0.5, 1.5], ['Normal', 'Fraud'])
plt.yticks([0.5, 1.5], ['Normal', 'Fraud'])
plt.show()

## 3. Feature Importance Analysis

In [ ]:
# Top feature importances
importances = pd.Series(dt_balanced.feature_importances_, index=X.columns).sort_values(ascending=False)
top_10_imp = importances.head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=top_10_imp.values, y=top_10_imp.index, palette='viridis')
plt.title('Top 10 Most Important Features in Decision Tree', fontsize=13, fontweight='bold')
plt.xlabel('Relative Importance (Gini Decrease)')
plt.show()

## 4. Visualizing the Decision Tree Structure

In [ ]:
# Visualize the top 3 levels of the tree
plt.figure(figsize=(20, 10))
plot_tree(
    dt_balanced,
    max_depth=3,
    feature_names=X.columns,
    class_names=['Normal', 'Fraud'],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title('Pruned Decision Tree (Top 3 Levels)', fontsize=14, fontweight='bold')
plt.show()